# Project_3

Dataset(s) to be used: [https://data.cityofnewyork.us/City-Government/Citywide-Payroll-Data-Fiscal-Year-/k397-673e/about_data]

Analysis question: [Is there a significant difference in base salary among employees working in different boroughs of New York City?]

Columns that will (likely) be used:

[work_location_borough]

[base_salary]

Hypothesis: [Employees working in Manhattan have significantly higher average base salaries than those in other boroughs, because Manhattan is the commercial and financial center of NYC with higher living costs.]

In [19]:
import pandas as pd
import pandas as pd
import numpy as np
import plotly.express as px

In [20]:
df = pd.read_csv('https://data.cityofnewyork.us/resource/k397-673e.csv')
df.head()

,fiscal_year,payroll_number,agency_name,last_name,first_name,mid_init,agency_start_date,work_location_borough,title_description,leave_status_as_of_june_30,base_salary,pay_basis,regular_hours,regular_gross_paid,ot_hours,total_ot_paid,total_other_pay
0,2025,67,ADMIN FOR CHILDREN'S SVCS,TAYLOR,DENISE,NaN,2006-09-04T00:00:00.000,MANHATTAN,SUPERVISOR II,ACTIVE,80632.0,per Annum,1820.0,78118.64,27.50,1522.72,5081.35
1,2025,67,ADMIN FOR CHILDREN'S SVCS,MENCIA,EDWIN,A,1998-03-09T00:00:00.000,MANHATTAN,EXECUTIVE ASSISTANT TO THE EXECUTIVE DEPUTY ADM,ACTIVE,142829.0,per Annum,1820.0,132898.25,0.00,0.00,5293.35
2,2025,67,ADMIN FOR CHILDREN'S SVCS,CALIM,SALIMIE,NaN,1996-06-23T00:00:00.000,MANHATTAN,CHILD PROTECTIVE SPECIALIST SUPERVISOR,CEASED,97327.0,per Annum,952.0,50922.84,0.00,0.00,2640.10
3,2025,67,ADMIN FOR CHILDREN'S SVCS,BIRKETT,LESLIE,NaN,1997-07-06T00:00:00.000,MANHATTAN,SUPERVISOR II,ACTIVE,80491.0,per Annum,1820.0,77981.47,0.00,0.00,5054.29
4,2025,67,ADMIN FOR CHILDREN'S SVCS,NEALY,DWAYNE,NaN,1996-06-23T00:00:00.000,MANHATTAN,CHILD PROTECTIVE SPECIALIST,ON SEPARATION LEAVE,70264.0,per Annum,1820.0,68070.52,300.75,15891.17,12663.10


In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   fiscal_year                 1000 non-null   int64  
 1   payroll_number              1000 non-null   int64  
 2   agency_name                 1000 non-null   object 
 3   last_name                   1000 non-null   object 
 4   first_name                  1000 non-null   object 
 5   mid_init                    538 non-null    object 
 6   agency_start_date           1000 non-null   object 
 7   work_location_borough       1000 non-null   object 
 8   title_description           1000 non-null   object 
 9   leave_status_as_of_june_30  1000 non-null   object 
 10  base_salary                 1000 non-null   float64
 11  pay_basis                   1000 non-null   object 
 12  regular_hours               1000 non-null   float64
 13  regular_gross_paid          1000 n

In [22]:
print("\nWork location borough distribution:")
borough_counts = df['work_location_borough'].value_counts()
print(borough_counts)


Work location borough distribution:
work_location_borough
MANHATTAN    623
BRONX        136
BROOKLYN     126
QUEENS        92
RICHMOND      23
Name: count, dtype: int64


In [23]:
print("\nBasic statistics for base salary:")
print(f"Average base salary: ${df['base_salary'].mean():,.2f}")
print(f"Median base salary: ${df['base_salary'].median():,.2f}")
print(f"Standard deviation: ${df['base_salary'].std():,.2f}")
print(f"Minimum base salary: ${df['base_salary'].min():,.2f}")
print(f"Maximum base salary: ${df['base_salary'].max():,.2f}")


Basic statistics for base salary:
Average base salary: $95,003.33
Median base salary: $93,598.00
Standard deviation: $33,461.86
Minimum base salary: $19.00
Maximum base salary: $280,644.00


In [24]:
# Calculate statistics for base salary by borough
borough_stats = df.groupby('work_location_borough')['base_salary'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(2)

borough_stats = borough_stats.rename(columns={
    'count': 'Employee Count',
    'mean': 'Average Base Salary',
    'median': 'Median Base Salary',
    'std': 'Standard Deviation',
    'min': 'Minimum Salary',
    'max': 'Maximum Salary'
})

print("Base salary statistics by borough:")
print(borough_stats)

Base salary statistics by borough:
                       Employee Count  Average Base Salary  \
work_location_borough                                        
BRONX                             136             89478.90   
BROOKLYN                          126             87645.32   
MANHATTAN                         623             97909.14   
QUEENS                             92             95260.98   
RICHMOND                           23             88238.22   

                       Median Base Salary  Standard Deviation  Minimum Salary  \
work_location_borough                                                           
BRONX                             81753.0            28890.70            31.9   
BROOKLYN                          79129.5            24445.78         39654.0   
MANHATTAN                         94716.0            36725.76            19.0   
QUEENS                            98860.0            26667.45         55741.0   
RICHMOND                          91388.0   

In [25]:
# Sort by average base salary
print("\nSorted by average base salary (descending):")
print(borough_stats.sort_values('Average Base Salary', ascending=False))


Sorted by average base salary (descending):
                       Employee Count  Average Base Salary  \
work_location_borough                                        
MANHATTAN                         623             97909.14   
QUEENS                             92             95260.98   
BRONX                             136             89478.90   
RICHMOND                           23             88238.22   
BROOKLYN                          126             87645.32   

                       Median Base Salary  Standard Deviation  Minimum Salary  \
work_location_borough                                                           
MANHATTAN                         94716.0            36725.76            19.0   
QUEENS                            98860.0            26667.45         55741.0   
BRONX                             81753.0            28890.70            31.9   
RICHMOND                          91388.0            21582.10         61376.0   
BROOKLYN                          

In [28]:
# Statistical testing - test if salary differences between boroughs are significant
# First, compare base salaries between Manhattan and other boroughs
manhattan_salaries = df[df['work_location_borough'] == 'MANHATTAN']['base_salary']
non_manhattan_salaries = df[df['work_location_borough'] != 'MANHATTAN']['base_salary']

print("Manhattan vs. Other Boroughs Salary Comparison:")
print(f"Manhattan average base salary: ${manhattan_salaries.mean():,.2f}")
print(f"Manhattan salary standard deviation: ${manhattan_salaries.std():,.2f}")
print(f"Manhattan employee count: {len(manhattan_salaries)}")
print(f"\nOther boroughs average base salary: ${non_manhattan_salaries.mean():,.2f}")
print(f"Other boroughs salary standard deviation: ${non_manhattan_salaries.std():,.2f}")
print(f"Other boroughs employee count: {len(non_manhattan_salaries)}")


Manhattan vs. Other Boroughs Salary Comparison:
Manhattan average base salary: $97,909.14
Manhattan salary standard deviation: $36,725.76
Manhattan employee count: 623

Other boroughs average base salary: $90,201.41
Other boroughs salary standard deviation: $26,581.90
Other boroughs employee count: 377


In [31]:
from scipy import stats

In [32]:
# Use t-test to check if the difference is significant
# Use independent samples t-test, assuming unequal variances (Welch's t-test)
t_stat, p_value = stats.ttest_ind(manhattan_salaries, non_manhattan_salaries, equal_var=False)
print(f"\nIndependent samples t-test results:")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("Conclusion: p-value < 0.05, the difference between Manhattan and other boroughs is statistically significant.")
else:
    print("Conclusion: p-value >= 0.05, the difference between Manhattan and other boroughs is not statistically significant.")


Independent samples t-test results:
t-statistic: 3.8351
p-value: 0.000134
Conclusion: p-value < 0.05, the difference between Manhattan and other boroughs is statistically significant.


In [35]:
# Calculate Manhattan salary premium relative to other boroughs
manhattan_mean = manhattan_salaries.mean()
non_manhattan_mean = non_manhattan_salaries.mean()
salary_premium = ((manhattan_mean - non_manhattan_mean) / non_manhattan_mean) * 100

print("Analysis Summary:")
print(f"1. Manhattan average base salary: ${manhattan_mean:,.2f}")
print(f"   Other boroughs average base salary: ${non_manhattan_mean:,.2f}")
print(f"2. Manhattan salary premium: {salary_premium:.2f}% higher than other boroughs")
print(f"3. Statistical test shows the difference is {'significant' if p_value < 0.05 else 'not significant'} (p = {p_value:.6f})")

Analysis Summary:
1. Manhattan average base salary: $97,909.14
   Other boroughs average base salary: $90,201.41
2. Manhattan salary premium: 8.55% higher than other boroughs
3. Statistical test shows the difference is significant (p = 0.000134)


In [39]:
print("\nHypothesis Testing:")
if p_value < 0.05 and manhattan_mean > non_manhattan_mean:
    print("✓ Accept the hypothesis: Employees in Manhattan have significantly higher average base salaries than those in other boroughs")
else:
    print("✗ Reject the hypothesis: Data does not support that Manhattan employees have significantly higher base salaries")


Hypothesis Testing:
✓ Accept the hypothesis: Employees in Manhattan have significantly higher average base salaries than those in other boroughs


In [40]:
# Statistical testing - Data Visualization

# Prepare the data needed
borough_means = df.groupby('work_location_borough')['base_salary'].mean().sort_values(ascending=False)
borough_means_df = borough_means.reset_index()
borough_means_df.columns = ['Borough', 'Average Base Salary']

# Prepare a bar chart
fig1 = px.bar(borough_means_df,
              x='Borough',
              y='Average Base Salary',
              title='Average Base Salary by Borough',
              color='Average Base Salary',
              color_continuous_scale='blues',
              text='Average Base Salary')

fig1.update_traces(texttemplate='$%{text:,.0f}',
                   textposition='outside',
                   marker_line_color='black',
                   marker_line_width=1)
fig1.update_layout(xaxis_title='Borough',
                   yaxis_title='Average Base Salary ($)',
                   xaxis_tickangle=-45,
                   coloraxis_showscale=False)
fig1.show()


In [45]:
# Create a new column to distinguish between Manhattan and other boroughs
df['location_category'] = df['work_location_borough'].apply(
    lambda x: 'MANHATTAN' if x == 'MANHATTAN' else 'OTHER BOROUGHS'
)

# Box plot of base salary comparison: Manhattan vs other boroughs

fig5 = px.box(df,
              x='location_category',
              y='base_salary',
              title='Manhattan vs Other Boroughs: Base Salary Comparison',
              color='location_category',
              points="outliers")

fig5.update_layout(xaxis_title='Location Category',
                   yaxis_title='Base Salary ($)',
                   showlegend=False)
fig5.show()